# 2 · Observability: reading the traces

Every turn the assistant serves is recorded as a trace. This notebook teaches
you to read one and answer the question that comes up on every agent project:
**why did it do that?**

| | |
|---|---|
| **Time** | about 30 minutes |
| **You need** | the stack from [1 · Getting Started](1_Getting_Started.ipynb), with Phoenix running |
| **Reference** | [docs/OBSERVABILITY.md](../docs/OBSERVABILITY.md) |

**You will learn to:**

1. Read a conversation one line per turn.
2. Open one turn and see every model call and tool call in it.
3. Answer four questions: which skill ran, what the model was told, what a tool
   returned, and why a call was refused.
4. Tell "the model chose badly" apart from "the system resolved badly".
5. See where a turn's time and tokens go.
6. Compare two runs of the same conversation.
7. Decide when to switch on NeMo Relay.

**How to use this notebook.** Run the cells in order. Each step says what to
**Run** or **Do**, then shows what **You should see**. Replies vary between
runs, but the shape of each output should match.

## 1. Three levels: session, trace, span

| Level | Is | Keyed by |
|---|---|---|
| **session** | one conversation | `session.id`, the `conversation_id` the client sends |
| **trace** | one turn | one trace id per request |
| **span** | one step in a turn | a model call, a tool call, the turn itself |

A Phoenix session and a conversation stored by the memory service are the same
set of turns, because both are grouped by `conversation_id`. What you read in a
trace is what the shopper had.

Two layers write spans, and a third is optional. All of them go through the
OpenTelemetry collector to Phoenix:

```
chain-server ──┬── turn span ─────────────────┐
               ├── openinference-langchain ───┼── otel-collector ──> Phoenix
               └── NeMo Relay (optional) ─────┘       :4318            :6006
```

| Layer | On | Gives you |
|---|---|---|
| **turn span** | whenever tracing is on | one span per turn: skills, tools, refused calls, products shown, how it ended |
| **openinference-langchain** | whenever tracing is on | every model call with its full prompt and reply, every tool call with its arguments and result |
| **NeMo Relay** | opt-in | spans named per model, with finish reason, plus lifecycle events. See [step 10](#10.-NeMo-Relay:-when-to-switch-it-on) |

## 2. Check that traces are flowing

Tracing is on when `OTEL_EXPORTER_OTLP_ENDPOINT` is set. `.env.example` sets it
to `http://otel-collector:4318`.

**Run:**

In [ ]:
import json, difflib, subprocess
from collections import Counter, defaultdict
from helpers import *

print("Phoenix     :", status(f"{PHOENIX}/v1/projects/default/spans?limit=1"))
print("chain-server:", status(f"{CHAIN_SERVER}/health"))
env = subprocess.run(["docker", "exec", "chain-server", "printenv"], capture_output=True, text=True).stdout
for name in ("OTEL_EXPORTER_OTLP_ENDPOINT", "RELAY_ENABLED"):
    print(f"{name}:", next((l.split("=", 1)[1] for l in env.splitlines() if l.startswith(name + "=")), "(not set)"))

**You should see** both services answering `200`, and the collector address:

```
Phoenix     : 200
chain-server: 200
OTEL_EXPORTER_OTLP_ENDPOINT: http://otel-collector:4318
RELAY_ENABLED: false
```

**If not:** if the endpoint shows `(not set)`, add it to `.env`, then run
`source .env && docker compose up -d chain-server` from the repo root. The
Phoenix UI is at [http://localhost:6006](http://localhost:6006). This notebook
reads the same data through Phoenix's REST API.

## 3. Make a conversation to study

A conversation that browses, adds to the cart, and reads the cart touches
everything this notebook covers.

**Run** the next cell. It empties the notebook shopper's cart, sends three
turns under a new conversation id, then waits for their traces to reach
Phoenix. The turns call the model.

In [ ]:
SCRIPT = [
    "show me black dresses in a size 2",
    "add the first one to my cart",
    "what's in my cart?",
]

def converse(script=SCRIPT):
    post(f"{MEMORY}/user/{NOTEBOOK_USER_ID}/clear", {})   # start from an empty cart
    conversation = new_conversation("obs")
    for text in script:
        print(f"> {text}\n  {say(text, conversation)['reply'][:120]}\n")
    return conversation

SESSION = converse()
SPANS = wait_for_turns(SESSION, len(SCRIPT))
TRACES = by_trace(SPANS)
TURN_SPANS = [r for r in SPANS if r["name"] == "turn"]

**You should see** each message with the start of its reply, then a count of
the turns that reached Phoenix:

```
> show me black dresses in a size 2
  Here are four black dresses in size 2: 1. **Black Satin Lace-Up Dress** ...

> add the first one to my cart
  Done — I added the **Black Satin Lace-Up Dress** in size 2 to your cart. ...

> what's in my cart?
  Your cart holds one item: - **Black Satin Lace-Up Dress** (size 2) — $69.99 ...

3 of 3 turns of obs-fdedd057 in Phoenix
```

**To study a conversation that already exists instead**, such as an evaluation
replay, set `SESSION` to its id (the Sessions tab in Phoenix lists them). Then
load more history with `SPANS = load_spans(max_pages=10)` and rebuild
`TRACES` and `TURN_SPANS` as above.

## 4. Read the conversation, one line per turn

Most questions ("why did it add the wrong dress?") are about a conversation,
not a single turn, so start here. Each line comes from the turn span. The
shopper's words come from the `LangGraph` span in the same trace.

**Run:**

In [ ]:
def turns_of(session_id):
    turns = [s for s in TURN_SPANS if s["attributes"].get("session.id") == session_id]
    return sorted(turns, key=lambda s: s["start_time"])

def said(turn):
    graph = next((s for s in TRACES[turn["context"]["trace_id"]] if s["name"] == "LangGraph"), None)
    try:
        message = json.loads(graph["attributes"]["input.value"])["messages"][-1]["data"]["content"]
    except (TypeError, KeyError, IndexError, ValueError):
        return "?"
    marker = "USER QUERY:"
    return message.split(marker, 1)[1].strip().splitlines()[0] if marker in message else message[-80:]

def read_session(session_id):
    for number, turn in enumerate(turns_of(session_id), 1):
        a = turn["attributes"]
        skills = ", ".join(s.split("/")[-2] for s in as_list(a.get("metadata.skills")))
        print(f'{number}. "{said(turn)}"')
        print(f"   skills {skills or '-'}")
        print(f"   tools  {', '.join(as_list(a.get('metadata.tools'))) or '-'}")
        print(f"   ->     {a.get('metadata.products_shown')} shown, "
              f"{a.get('metadata.tool_calls_rejected')} refused, {a.get('metadata.termination_reason')}\n")

read_session(SESSION)

**You should see** one block per turn:

```
1. "show me black dresses in a size 2"
   skills product-discovery
   tools  activate_shopper_skills_tool, search_catalog_tool
   ->     4 shown, 0 refused, completed

2. "add the first one to my cart"
   skills cart-management
   tools  activate_shopper_skills_tool, add_cart_items_tool
   ->     0 shown, 0 refused, completed

3. "what's in my cart?"
   skills cart-management
   tools  activate_shopper_skills_tool, get_cart_tool
   ->     0 shown, 0 refused, completed
```

**How to read each field:**

| Field | Tells you |
|---|---|
| `skills` | which skill was chosen. A rule in a skill that never loaded is the most common root cause here |
| `tools` | what actually ran. A `search_catalog_tool` on "add the first one" means the agent did not recognise a product it had shown |
| `shown` | products streamed to the shopper. `0` on a discovery turn is either an honest "we don't carry that" or a failure; the reply says which |
| `refused` | tool calls the server turned down. One is the gates working. Several means the model kept retrying something it may not do |
| last word | how the turn ended. Anything but `completed` (`timeout`, `max_iterations`) means the reply is partial |

The same view is in Phoenix's **Sessions** tab, and on the command line:
`python3 scripts/read_session.py <session> --replies`.

## 5. Open one turn

Once the conversation view points at a turn, open its trace.

**Run** the next cell. It opens the cart-add turn and prints every span,
nested under its parent.

In [ ]:
TURN_NUMBER = 2
TURN = turns_of(SESSION)[TURN_NUMBER - 1]
TRACE = TRACES[TURN["context"]["trace_id"]]

def tree(trace):
    children = defaultdict(list)
    for span in trace:
        children[span["parent_id"]].append(span)
    ids = {s["context"]["span_id"] for s in trace}
    def walk(span, depth):
        print(f"{'  ' * depth}{span['span_kind']:6} {span['name']:<38} {seconds(span):6.2f}s")
        for child in children[span["context"]["span_id"]]:
            walk(child, depth + 1)
    for root in (s for s in trace if s["parent_id"] not in ids):
        walk(root, 0)

print(f'Turn {TURN_NUMBER}: "{said(TURN)}"  ({len(TRACE)} spans)\n')
tree(TRACE)

**You should see** a tree like this:

```
Turn 2: "add the first one to my cart"  (14 spans)

CHAIN  turn                                     6.35s
  CHAIN  LangGraph                                5.34s
    AGENT  PatchToolCallsMiddleware.before_agent    0.00s
    CHAIN  model                                    1.82s
      LLM    ChatOpenAI                               1.78s
    CHAIN  tools                                    0.00s
      TOOL   activate_shopper_skills_tool             0.00s
    CHAIN  model                                    2.22s
      LLM    ChatOpenAI                               2.21s
    CHAIN  tools                                    0.05s
      TOOL   add_cart_items_tool                      0.04s
    CHAIN  model                                    1.24s
      LLM    ChatOpenAI                               1.23s
  LLM    ChatOpenAI                               0.96s
```

**How to read it:**

- **`turn`** is the whole turn and carries its diagnostics.
- **`LangGraph`** is the agent loop. Each **`model` → `ChatOpenAI`** is the
  model deciding, and each **`tools` → `<tool>`** is the tool it chose. One
  pair is one round.
- The **`ChatOpenAI` directly under `turn`** is the **grounding editor**. It
  rewrites the draft reply so that every claim is backed by tool evidence, the
  cart, or products shown earlier.

Counting rounds is the quickest read. Three rounds to add one product is
normal: choose the skill, add, then write the reply. Nine means the agent was struggling, and the tool results between
the rounds say why.

**Run:**

In [ ]:
loop_calls = [s for s in TRACE if s["name"] == "ChatOpenAI"
              and any(p["name"] == "model" and p["context"]["span_id"] == s["parent_id"] for p in TRACE)]
tool_calls = [s for s in TRACE if s["span_kind"] == "TOOL"]
print("agent-loop model calls:", len(loop_calls))
print("tool calls:", [s["name"] for s in tool_calls])

**You should see:**

```
agent-loop model calls: 3
tool calls: ['activate_shopper_skills_tool', 'add_cart_items_tool']
```

## 6. The four questions

### Q1. Which skill ran, and what did the agent do?

The turn span answers this. `metadata.diagnostics_json` is the full record:
every tool call in order, with its arguments and status.

**Run:**

In [ ]:
a = TURN["attributes"]
for key in ("metadata.skills", "metadata.tools", "metadata.tool_calls_rejected",
            "metadata.products_shown", "metadata.termination_reason"):
    print(f"{key:30} {a.get(key)}")

DIAGNOSTICS = json.loads(a.get("metadata.diagnostics_json") or "{}")
print("\ndiagnostics_json:", ", ".join(DIAGNOSTICS))
for call in DIAGNOSTICS.get("tool_calls", []):
    print(f"  {call['sequence']}. {call['tool_name']:36} {call['status']:10} {json.dumps(call.get('arguments'))[:70]}")

**You should see** the summary fields, the sections of the record, then each
call with the start of its arguments:

```
metadata.skills                ['/shopper/cart-management/SKILL.md']
metadata.tools                 ['activate_shopper_skills_tool', 'add_cart_items_tool']
metadata.tool_calls_rejected   0
metadata.products_shown        0
metadata.termination_reason    completed

diagnostics_json: skill_files_read, tool_calls, rejected_tool_calls, duplicate_tool_calls, product_evidence, ...
  1. activate_shopper_skills_tool         completed  {"skill_names": ["cart-management"]}
  2. add_cart_items_tool                  completed  {"items": [{"product_ref": "generated:3185c59c1cab8b83", ...
```

### Q2. What was the model actually told?

Every `ChatOpenAI` span holds the full conversation as sent: the system prompt
(in several content blocks), the shopper's turn, each tool call the model made,
and each tool result it saw. This is how you tell **"the model ignored the
rule"** from **"the model never saw the rule"**.

**Run** the next cell. It lists what the last round of the loop was sent.

In [ ]:
def message_text(a, base):
    parts = [a.get(f"{base}.content", "").strip()]
    i = 0
    while f"{base}.contents.{i}.message_content.text" in a:
        parts.append(a[f"{base}.contents.{i}.message_content.text"])
        i += 1
    i = 0
    while f"{base}.tool_calls.{i}.tool_call.function.name" in a:
        name = a[f"{base}.tool_calls.{i}.tool_call.function.name"]
        parts.append(f"CALLS {name}({a.get(f'{base}.tool_calls.{i}.tool_call.function.arguments', '')})")
        i += 1
    return "\n".join(p for p in parts if p)

def messages(span, prefix="llm.input_messages"):
    a = span["attributes"]
    count = len({k.split(".")[2] for k in a if k.startswith(prefix + ".")})
    return [(a.get(f"{prefix}.{i}.message.role"), message_text(a, f"{prefix}.{i}.message"))
            for i in range(count)]

last_round = loop_calls[-1]
for role, content in messages(last_round):
    print(f"[{role:9}] {len(content):6} chars | {content[:110].replace(chr(10), ' ')}")

**You should see** the system prompt first, then the conversation, including
the model's own tool calls (`CALLS ...`) and the tool results:

```
[system   ]  31624 chars | You are a retail shopping assistant for the products advertised by the active catalog. ...
[user     ]   2473 chars | REQUEST ID: ... SESSION ID: obs-fdedd057 CONVERSATION ID: obs-fdedd057 CART ...
[assistant]     72 chars | CALLS activate_shopper_skills_tool({"skill_names": ["cart-management"]})
[tool     ]     68 chars | SHOPPER_SKILL_ACTIVATION_COMPLETE: /shopper/cart-management/SKILL.md
[assistant]    152 chars | CALLS add_cart_items_tool({"items": [{"product_ref": "generated:3185c59c1cab8b83", ...
[tool     ]    517 chars | CART_ADD_RESULT Added: - 1 x Black Satin Lace-Up Dress, size 2 ...
```

**Run** the next cell to check whether a rule reached the model. Set `RULE` to
any phrase from a skill or the prompt.

In [ ]:
RULE = "CART_LINE_ID"
for number, span in enumerate(loop_calls, 1):
    hits = [role for role, content in messages(span) if RULE in content]
    print(f"round {number}:", f"seen in {', '.join(hits)}" if hits else "NOT in what the model was sent")

**You should see** which messages carried the phrase in each round:

```
round 1: NOT in what the model was sent
round 2: seen in system
round 3: seen in system, tool
```

Here the phrase comes from the cart skill, so the model sees it only after the
skill loads in round 1. A rule missing from a round is never a model bug: find
where the rule should have come from.

### Q3. What did a tool return?

In a tool span, `input.value` is what the model asked for, and `output.value`
is what came back.

**Run:**

In [ ]:
def result(span):
    # output.value is the tool message as JSON; its content is what the model read.
    raw = span["attributes"].get("output.value") or ""
    try:
        return json.loads(raw)["data"]["content"]
    except (ValueError, KeyError, TypeError):
        return raw

for span in tool_calls:
    print(f"== {span['name']}")
    print("   asked:", str(span["attributes"].get("input.value"))[:300])
    print("   got  :", result(span)[:400], "\n")

**You should see** each tool's arguments and result. For
`add_cart_items_tool`, the result says exactly what went into the cart, and
whether the cart already held the product or another size of it:

```
== activate_shopper_skills_tool
   asked: {"skill_names": ["cart-management"]}
   got  : SHOPPER_SKILL_ACTIVATION_COMPLETE: /shopper/cart-management/SKILL.md

== add_cart_items_tool
   asked: {"items": [{"product_ref": "generated:3185c59c1cab8b83", "expected_display_name": "Black Satin Lace-Up Dress", "size": "2"}]}
   got  : CART_ADD_RESULT
Added:
- 1 x Black Satin Lace-Up Dress, size 2 (PRODUCT_REF: generated:3185c59c1cab8b83)
Current cart:
- CART_LINE_ID: c45e4bd6c7a44961a0d11f815f48329d | 1 x Black Satin Lace-Up Dress (size 2) @ $69.99
...
```

### Q4. Why was a tool call refused?

A refused call has `status: rejected` in the diagnostics, with a
`rejection_reason`. Examples: `skill_activation_required`,
`skill_tool_not_granted`, `duplicate_catalog_scope`. These gates exist so that a
wrong tool call cannot turn into a wrong answer.

**Run** the next cell. It scans the whole conversation.

In [ ]:
refused = 0
for number, turn in enumerate(turns_of(SESSION), 1):
    record = json.loads(turn["attributes"].get("metadata.diagnostics_json") or "{}")
    for call in record.get("tool_calls") or []:
        if call.get("status") == "rejected":
            refused += 1
            print(f"turn {number}, call {call['sequence']}: {call['tool_name']} -> {call.get('rejection_reason')}")
print(refused, "refused call(s)")

**You should see** `0 refused call(s)` for a smooth conversation. To see one,
**Try** a conversation that starts as browsing and ends by asking for a cart
change, for example `converse(["show me heels", "actually just add the first one"])`,
and read its diagnostics.

## 7. Worked example: "it added the wrong product"

This sequence separates **the model chose badly** from **the system resolved
badly**:

1. **Read the add call's `asked`.** `product_ref` is the product the model
   chose, and `expected_display_name` is what it believed that product to be.
2. **No resolver call:** the model picked the product itself, from the numbered
   products it was shown ("the first one" works this way). If the choice is
   wrong, the model misread the shopper. The round's `ChatOpenAI` input (Q2)
   shows the list it chose from.
3. **A `resolve_conversation_products_tool` call:** the model described the
   product ("the black one") and the resolver matched it. A wrong description
   is the model's mistake. A right description with a wrong match is the
   resolver's.
4. **Read the add call's `got`.** It shows what was actually added, and says
   whether the cart already held that product or another size of it.

**Run:**

In [ ]:
CART_WRITES = {"add_cart_items_tool", "update_cart_items_tool", "remove_cart_item_tool"}

def explain_cart_turn(session_id, turn_number):
    turn = turns_of(session_id)[turn_number - 1]
    trace = TRACES[turn["context"]["trace_id"]]
    names = [s["name"] for s in trace if s["span_kind"] == "TOOL"]
    print(f'Turn {turn_number}: "{said(turn)}"\n  tools: {names}\n')
    if "resolve_conversation_products_tool" not in names:
        print("  No resolver call: the model chose the product itself, from what it was shown.\n")
    for span in trace:
        if span["name"] in {"resolve_conversation_products_tool", *CART_WRITES}:
            print(f"  {span['name']}")
            print(f"    asked: {str(span['attributes'].get('input.value'))[:250]}")
            print(f"    got  : {result(span)[:250]}\n")

explain_cart_turn(SESSION, TURN_NUMBER)

**You should see** the turn's tools, how the product was chosen, then the cart
write:

```
Turn 2: "add the first one to my cart"
  tools: ['activate_shopper_skills_tool', 'add_cart_items_tool']

  No resolver call: the model chose the product itself, from what it was shown.

  add_cart_items_tool
    asked: {"items": [{"product_ref": "generated:3185c59c1cab8b83", "expected_display_name": "Black Satin Lace-Up Dress", "size": "2"}]}
    got  : CART_ADD_RESULT
Added:
- 1 x Black Satin Lace-Up Dress, size 2 (PRODUCT_REF: generated:3185c59c1cab8b83) ...
```

## 8. Where the time and tokens go

A turn's time is mostly model round trips.

**Run** the next cell. It splits each turn into model time, tool time, and the
rest, and totals the tokens.

In [ ]:
def cost(trace):
    turn = next(s for s in trace if s["name"] == "turn")
    llm = [s for s in trace if s["span_kind"] == "LLM"]
    tools = [s for s in trace if s["span_kind"] == "TOOL"]
    tokens = Counter()
    for span in llm:
        for kind in ("prompt", "completion"):
            tokens[kind] += int(span["attributes"].get(f"llm.token_count.{kind}") or 0)
    total, in_llm, in_tools = seconds(turn), sum(map(seconds, llm)), sum(map(seconds, tools))
    return {"turn_s": round(total, 1), "model_s": round(in_llm, 1), "tools_s": round(in_tools, 1),
            "other_s": round(total - in_llm - in_tools, 1), "model_calls": len(llm),
            "prompt_tok": tokens["prompt"], "completion_tok": tokens["completion"]}

print("turn " + "".join(f"{k:>15}" for k in cost(TRACE)))
for number, turn in enumerate(turns_of(SESSION), 1):
    print(f"{number:>4} " + "".join(f"{v:>15}" for v in cost(TRACES[turn['context']['trace_id']]).values()))

**You should see** one row per turn:

```
turn          turn_s        model_s        tools_s        other_s    model_calls     prompt_tok completion_tok
   1            13.1           10.0            2.8            0.3              4          33091            587
   2             6.3            6.2            0.0            0.1              4          37979            271
   3             7.4            7.3            0.0            0.1              4          38223            156
```

Prompt tokens grow with every round, because each model call re-sends the
conversation and every tool result so far. A turn with many rounds costs twice:
more calls, and larger ones. `model_s` includes the grounding editor.

## 9. Compare two runs of the same conversation

To see what changed between two runs, compare their **shape** (the skills and
tools of each turn), not their wording. A turn that chose a different skill, or
stopped calling the resolver, stands out at once.

**Run** the next cell. It holds the same conversation again, from an empty
cart, then compares the two.

In [ ]:
def shape(session_id):
    return [f"{n} [{','.join(s.split('/')[-2] for s in as_list(t['attributes'].get('metadata.skills')))}] "
            f"{' > '.join(as_list(t['attributes'].get('metadata.tools')))}"
            for n, t in enumerate(turns_of(session_id), 1)]

RUN_A, RUN_B = SESSION, converse()
SPANS = wait_for_turns(RUN_B, len(SCRIPT))
TRACES, TURN_SPANS = by_trace(SPANS), [r for r in SPANS if r["name"] == "turn"]

diff = list(difflib.unified_diff(shape(RUN_A), shape(RUN_B), RUN_A, RUN_B, lineterm="", n=0))
print("\n".join(diff) or "Same shape: every turn chose the same skills and ran the same tools.")

**You should see** either `Same shape: ...`, or the turns that differ:

```
--- obs-fdedd057
+++ obs-15306297
@@ -1 +1 @@
-1 [product-discovery] activate_shopper_skills_tool > search_catalog_tool
+1 [outfit-styling] activate_shopper_skills_tool > search_catalog_tool
```

A different skill for the same words is worth investigating, even when both
replies look fine. Evaluation, in notebook 3, runs this comparison over many
conversations and repeats.

## 10. NeMo Relay: when to switch it on

Everything above ran **without** Relay. The two always-on layers already hold
the full prompt of every model call, the arguments and result of every tool
call, the loop structure, the timings, and the token counts.

**Relay adds:**

- one span per model call, named for the model, with its **finish reason**
  (useful when a reply was cut off by a token limit);
- lifecycle `mark:` events from the agent runtime.

**Relay does not:**

- **join the turn's trace.** One turn arrives as about seven separate traces,
  because Relay's model and tool calls run outside its own scope;
- **say which skill ran.** Skills reach the agent through files, not Relay, so
  keep using `metadata.skills` on the turn span.

**Do**, to study a specific problem, then switch it off again:

```bash
# in .env
export INSTALL_RELAY=true     # read at build: puts the library in the image
export RELAY_ENABLED=true     # read at start: switches it on
```

```bash
source .env
docker compose build chain-server && docker compose up -d chain-server
docker logs chain-server | grep -i relay     # Relay tracing enabled, exporting to ...
```

Set both back to `false` and rebuild when you are done, so the image stops
carrying a library that can export prompts and cart contents.

**Run** the next cell to check whether Relay is exporting. Traces alone prove
nothing, because the LangChain layer produces them without Relay.

In [ ]:
relay_spans = [r for r in SPANS if any(k.startswith("nemo_relay.") for k in r["attributes"])]
print("Relay spans:", len(relay_spans))
print(sorted({r["name"] for r in relay_spans})[:10] if relay_spans else "Relay is off, not installed, or not exporting.")

**You should see**, with the default settings:

```
Relay spans: 0
Relay is off, not installed, or not exporting.
```

With Relay on, about six spans arrive per turn. **Zero while turn spans keep
arriving** is the failure to look for. `docker logs chain-server | grep -i relay`
says why, and [docs/OBSERVABILITY.md](../docs/OBSERVABILITY.md) maps each
message to its cause.

**Relay cannot change what the agent does.** It hooks the agent's middleware,
which could in principle alter a call. The runtime guarantees
(`_relay_may_observe_but_not_decide` in `chain_server/src/runtime/runtime.py`)
that every model and tool call receives the request that arrived, runs exactly
once, and returns its own result to the agent. If Relay's wrapping looks wrong,
the runtime logs a warning and builds the agent without it. Tracing degrades;
the shop does not change.

## 11. Exercises

1. **Find the slowest turn** with `cost()`. Is it slow because of many rounds, one slow tool, or the grounding editor?
2. **Prove a rule reached the model.** Take a line from `chain_server/skills/shopper/cart-management/SKILL.md`, set it as `RULE` in step 6, and check a cart turn. Then check a browsing turn. Should it be there?
3. **Separate model from system.** Run `converse(["show me black heels", "add the black one in size 8"])`. Did the resolver run? Decide whether a wrong result would have come from the model's description or from the resolver's match.
4. **Watch the grounding editor.** Compare the input and output of the `ChatOpenAI` span directly under `turn`. What did it change, and why?
5. **Provoke a refusal.** Run the conversation from Q4, then find the refused call and its reason.

## 12. Retention and privacy

Phoenix stores prompts, replies, and tool results, which is more than the memory
service keeps. They live in the `phoenix-data` volume. Compose publishes Phoenix
on all interfaces so a tunnel can reach it. On anything other than a development
machine, bind it to `127.0.0.1` and reach it through the tunnel.

Only live turns are traced. There is no backfill.

**Next: 3 · Evaluation** (coming next) measures this behaviour across many
conversations.